# fishock：用 Colab 免費 GPU 做音樂、環境音、音效、角色語音

**怎麼用**
1. 上方選單「執行階段 → 變更執行階段類型」，選 **T4 GPU**，儲存。
2. 從上到下，一格一格按左邊的 ▶ 執行。每一格跑完（左邊出現 ✓）再按下一格。
3. 最後一格會把成品打包成 zip 下載到你的電腦，傳給 Claude 就好。

提示詞都在 `prompts.json`，想改內容就改那個檔，不用改這裡。

> 這份筆記本還沒在 Colab 上實際跑過，如果某一格出錯，把錯誤訊息截圖給 Claude。
> 免費 Colab 一次大約能用幾個小時 GPU，不夠的話隔天再繼續。

## 1. 確認有 GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 下載 fishock 的腳本和提示詞

In [ ]:
%cd /content
!rm -rf fishock
!git clone --depth 1 --filter=blob:none --sparse https://github.com/brian18964-bot/fishock.git
%cd /content/fishock
!git sparse-checkout set tools/ai_audio
%cd /content/fishock/tools/ai_audio
!ls

## 3. 背景音樂（ACE-Step 1.5）
裝好之後會在背景開一個伺服器，再用 `gen_music.py` 生成三首，每首 3 個版本。第一次要下載模型，約 10–20 分鐘。

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git
!pip install -q uv
%cd /content/ACE-Step-1.5
!uv sync 2>&1 | tail -5
!ls /content/ACE-Step-1.5 | head

In [ ]:
import subprocess, time, urllib.request, os
ACE = "/content/ACE-Step-1.5"
if not os.path.isdir(ACE):
    raise SystemExit("找不到 ACE-Step-1.5 資料夾：上一格（安裝）沒有成功，請重新執行上一格，把最後的訊息截圖")
os.environ["ACESTEP_CONFIG_PATH"] = "acestep-v15-turbo"
os.environ["ACESTEP_LM_MODEL_PATH"] = "acestep-5Hz-lm-0.6B"
log = open("/content/acestep.log", "w")
server = subprocess.Popen(["uv", "run", "acestep-api"], cwd=ACE, stdout=log, stderr=subprocess.STDOUT)
ok = False
for i in range(90):  # 最多等 15 分鐘（第一次要下載模型）
    if server.poll() is not None:
        print("伺服器啟動失敗（程式結束了，代碼 %d）" % server.returncode); break
    try:
        urllib.request.urlopen("http://localhost:8001/docs", timeout=2)
        ok = True; print("伺服器好了"); break
    except Exception:
        time.sleep(10)
        if i % 6 == 5: print("還在準備中…（%d 分鐘）" % ((i + 1) // 6))
if not ok:
    print("---- 記錄的最後一段 ----")
    print(open("/content/acestep.log").read()[-3000:])

In [ ]:
%cd /content/fishock/tools/ai_audio
!python gen_music.py --candidates 3

In [ ]:
# 聽聽看
from IPython.display import Audio, display
import glob
for f in sorted(glob.glob("/content/fishock/tools/ai_audio/out/raw/music_*/*.wav")):
    print(f); display(Audio(f))

In [ ]:
# 音樂做完了，關掉伺服器讓出 GPU 記憶體
server.terminate(); time.sleep(5)
!pkill -f acestep-api || true

## 4. 環境音與音效（MOSS-SoundEffect v2）
裝在獨立的環境裡（套件版本跟上面的會衝突）。9 種環境音＋23 種音效，每種 3 個版本，大約要 1 小時。
只想先做環境音，把最後一行改成 `--kind ambience`。

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/OpenMOSS/MOSS-TTS.git
!python -m venv /content/env_moss
!/content/env_moss/bin/pip install -q --upgrade pip
%cd /content/MOSS-TTS/moss_soundeffect_v2
!/content/env_moss/bin/pip install -q --extra-index-url https://download.pytorch.org/whl/cu128 -e ".[torch-cu128]" 2>&1 | tail -3

In [ ]:
%cd /content/fishock/tools/ai_audio
!/content/env_moss/bin/python gen_sfx.py --candidates 3

## 5. 角色語音（Qwen3-TTS）
先設計聲音（每個角色 4 個樣本），聽完挑最像的，再用挑好的聲音念台詞。

In [ ]:
!python -m venv /content/env_tts
!/content/env_tts/bin/pip install -q --upgrade pip
!/content/env_tts/bin/pip install -q -U qwen-tts soundfile 2>&1 | tail -3
%cd /content/fishock/tools/ai_audio
!/content/env_tts/bin/python gen_voice.py design

In [ ]:
from IPython.display import Audio, display
import glob
for f in sorted(glob.glob("/content/fishock/tools/ai_audio/out/voice_ref/*.wav")):
    print(f); display(Audio(f))

In [ ]:
# 把下面兩個數字改成你挑的樣本編號（1-4），再執行
MERCHANT = 1
WILLOW = 1
%cd /content/fishock/tools/ai_audio
!/content/env_tts/bin/python gen_voice.py speak --ref merchant=out/voice_ref/merchant_{MERCHANT}.wav --ref willow=out/voice_ref/willow_{WILLOW}.wav

## 6. 打包下載（全部候選版本，Claude 會幫忙挑、整理）

In [ ]:
%cd /content/fishock/tools/ai_audio
!zip -qr /content/fishock_audio.zip out
from google.colab import files
files.download("/content/fishock_audio.zip")